# Noun Phrase Chunking using HMM
## Forward Algorithm + Viterbi Decoding + Add-K Smoothing + Seqeval

This notebook implements the complete assignment using the uploaded `train_NP.txt` and `test_NP.txt` files.

**Tasks covered**
1. Train an HMM from the training data.
2. Use Add-K smoothing for zero counts.
3. Use the Forward Algorithm to find the probability of every test sentence.
4. Use Viterbi decoding to predict `B-NP`, `I-NP`, and `O`.
5. Evaluate predictions using `seqeval`.
6. Save forward probabilities and Viterbi predictions.


In [3]:
!pip install -U setuptools==68.2.2 setuptools-scm==7.1.0 -q
!pip install seqeval --no-cache-dir --use-pep517 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import math
import os
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

from seqeval.metrics import (
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

print("Libraries imported successfully!")

In [ ]:
# Check uploaded files
print(os.listdir("/content"))

In [ ]:
def read_data(filename):
    sentences = []
    words = []
    tags = []

    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            # Blank line separates sentences
            if not line:
                if words:
                    sentences.append((words, tags))
                    words = []
                    tags = []
            else:
                parts = line.split("\t")
                if len(parts) != 2:
                    raise ValueError(f"Unexpected line format: {line}")

                word, tag = parts
                words.append(word)
                tags.append(tag)

    if words:
        sentences.append((words, tags))

    return sentences


TRAIN_FILE = "/content/train_NP.txt"
TEST_FILE = "/content/test_NP.txt"

train_data = read_data(TRAIN_FILE)
test_data = read_data(TEST_FILE)

print("Training sentences:", len(train_data))
print("Test sentences:", len(test_data))

In [ ]:
# Inspect first training sentence
words, tags = train_data[0]

print("First training sentence:")
for word, tag in zip(words, tags):
    print(f"{word:20s} {tag}")

In [ ]:
# Tags and vocabulary
tags = sorted(set(
    tag
    for sentence in train_data
    for tag in sentence[1]
))

vocab = set(
    word
    for sentence in train_data
    for word in sentence[0]
)

print("Tags:", tags)
print("Number of tags:", len(tags))
print("Vocabulary size:", len(vocab))

In [ ]:
# Initial tag counts
initial_counts = Counter()

for words, sentence_tags in train_data:
    initial_counts[sentence_tags[0]] += 1

print("Initial counts:")
for tag in tags:
    print(f"{tag:5s}: {initial_counts[tag]}")

In [ ]:
# Transition counts and tag counts
transition_counts = Counter()
tag_counts = Counter()

for words, sentence_tags in train_data:
    for tag in sentence_tags:
        tag_counts[tag] += 1

    for i in range(1, len(sentence_tags)):
        previous_tag = sentence_tags[i - 1]
        current_tag = sentence_tags[i]
        transition_counts[(previous_tag, current_tag)] += 1

print("Tag counts:")
for tag in tags:
    print(f"{tag:5s}: {tag_counts[tag]}")

In [ ]:
# Emission counts
emission_counts = Counter()

for words, sentence_tags in train_data:
    for word, tag in zip(words, sentence_tags):
        emission_counts[(tag, word)] += 1

print("Number of emission pairs:", len(emission_counts))
print("\nFirst 20 emission counts:")
for item, count in list(emission_counts.items())[:20]:
    print(item, count)

In [ ]:
# Add-K smoothing
K = 0.1

num_tags = len(tags)
num_sentences = len(train_data)
vocab_size = len(vocab)

print("K =", K)
print("Number of tags =", num_tags)
print("Number of training sentences =", num_sentences)
print("Vocabulary size =", vocab_size)

In [ ]:
def initial_probability(tag):
    return (
        initial_counts[tag] + K
    ) / (
        num_sentences + K * num_tags
    )


def transition_probability(previous_tag, current_tag):
    return (
        transition_counts[(previous_tag, current_tag)] + K
    ) / (
        tag_counts[previous_tag] + K * num_tags
    )


def emission_probability(word, tag):
    return (
        emission_counts[(tag, word)] + K
    ) / (
        tag_counts[tag] + K * vocab_size
    )


print("Initial probabilities:")
for tag in tags:
    print(tag, initial_probability(tag))

print("\nExample transition:")
print("P(I-NP | B-NP) =", transition_probability("B-NP", "I-NP"))

print("\nExample emission:")
print("P(patient | I-NP) =", emission_probability("patient", "I-NP"))

## 1. Forward Algorithm

For a sentence with words \(w_1,\ldots,w_n\), the forward probability is computed by summing over all possible tag sequences.

Initialization:

\[
\alpha_1(t)=P(t)P(w_1|t)
\]

Recursion:

\[
\alpha_i(t)=P(w_i|t)\sum_{t'}\alpha_{i-1}(t')P(t|t')
\]

Termination:

\[
P(W)=\sum_t \alpha_n(t)
\]

Log probabilities are used to avoid numerical underflow.


In [ ]:
def log_sum_exp(values):
    max_value = max(values)

    if max_value == -math.inf:
        return -math.inf

    total = sum(
        math.exp(value - max_value)
        for value in values
    )

    return max_value + math.log(total)


def forward_algorithm(words):
    alpha = {}

    # Initialization
    first_word = words[0]

    for tag in tags:
        alpha[tag] = (
            math.log(initial_probability(tag))
            + math.log(emission_probability(first_word, tag))
        )

    # Recursion
    for word in words[1:]:
        new_alpha = {}

        for current_tag in tags:
            values = []

            for previous_tag in tags:
                values.append(
                    alpha[previous_tag]
                    + math.log(
                        transition_probability(
                            previous_tag,
                            current_tag
                        )
                    )
                )

            new_alpha[current_tag] = (
                math.log(
                    emission_probability(word, current_tag)
                )
                + log_sum_exp(values)
            )

        alpha = new_alpha

    # Termination
    sentence_log_probability = log_sum_exp(
        list(alpha.values())
    )

    sentence_probability = math.exp(
        sentence_log_probability
    )

    return sentence_probability, sentence_log_probability

In [ ]:
# Test Forward Algorithm on the first test sentence
words, gold_tags = test_data[0]

probability, log_probability = forward_algorithm(words)

print("Sentence:")
print(" ".join(words))

print("\nProbability:")
print(probability)

print("\nLog probability:")
print(log_probability)

In [ ]:
# Forward probability for every test sentence
forward_results = []

for i, (words, gold_tags) in enumerate(test_data):
    probability, log_probability = forward_algorithm(words)

    forward_results.append({
        "sentence_number": i + 1,
        "sentence": " ".join(words),
        "probability": probability,
        "log_probability": log_probability
    })

print("Forward probabilities calculated!")
print("Number of test sentences:", len(forward_results))

In [ ]:
# Display first 10 forward results
for result in forward_results[:10]:
    print("=" * 80)
    print("Sentence", result["sentence_number"])
    print(result["sentence"])
    print("Probability:", result["probability"])
    print("Log Probability:", result["log_probability"])

In [ ]:
# Save forward probabilities
FORWARD_FILE = "/content/forward_probabilities.txt"

with open(FORWARD_FILE, "w", encoding="utf-8") as f:
    for result in forward_results:
        f.write(f"Sentence {result['sentence_number']}\n")
        f.write(result["sentence"] + "\n")
        f.write(f"Probability: {result['probability']}\n")
        f.write(f"Log Probability: {result['log_probability']}\n")
        f.write("\n")

print("Saved:", FORWARD_FILE)

## 2. Viterbi Decoding

Viterbi finds the single most probable tag sequence.

For each position and current tag, it stores:

- the best log probability;
- the previous tag that produced that best probability.

The backpointers are then followed backwards to recover the complete tag sequence.


In [ ]:
def viterbi(words):
    V = []
    backpointer = []

    # Initialization
    first_word = words[0]

    first_v = {}
    first_bp = {}

    for tag in tags:
        first_v[tag] = (
            math.log(initial_probability(tag))
            + math.log(emission_probability(first_word, tag))
        )
        first_bp[tag] = None

    V.append(first_v)
    backpointer.append(first_bp)

    # Recursion
    for word in words[1:]:
        current_v = {}
        current_bp = {}

        for current_tag in tags:
            best_score = -math.inf
            best_previous_tag = None

            for previous_tag in tags:
                score = (
                    V[-1][previous_tag]
                    + math.log(
                        transition_probability(
                            previous_tag,
                            current_tag
                        )
                    )
                    + math.log(
                        emission_probability(
                            word,
                            current_tag
                        )
                    )
                )

                if score > best_score:
                    best_score = score
                    best_previous_tag = previous_tag

            current_v[current_tag] = best_score
            current_bp[current_tag] = best_previous_tag

        V.append(current_v)
        backpointer.append(current_bp)

    # Termination
    best_final_tag = max(
        V[-1],
        key=V[-1].get
    )

    best_score = V[-1][best_final_tag]

    # Backtracking
    best_path = [best_final_tag]

    for i in range(len(words) - 1, 0, -1):
        best_path.append(
            backpointer[i][best_path[-1]]
        )

    best_path.reverse()

    return best_path, best_score

In [ ]:
# Test Viterbi on the first test sentence
words, gold_tags = test_data[0]

predicted_tags, score = viterbi(words)

print("Word                 Gold       Predicted")
print("-" * 55)

for word, gold, pred in zip(words, gold_tags, predicted_tags):
    print(f"{word:20s} {gold:10s} {pred}")

print("\nViterbi log score:", score)

In [ ]:
# Viterbi decoding for the complete test set
all_gold_tags = []
all_predicted_tags = []
viterbi_results = []

for words, gold_tags in test_data:
    predicted_tags, score = viterbi(words)

    all_gold_tags.append(gold_tags)
    all_predicted_tags.append(predicted_tags)

    viterbi_results.append({
        "words": words,
        "gold_tags": gold_tags,
        "predicted_tags": predicted_tags,
        "score": score
    })

print("Viterbi decoding completed!")
print("Test sentences:", len(viterbi_results))

In [ ]:
# Display first 3 predicted sentences
for sentence_number, result in enumerate(viterbi_results[:3], start=1):
    print("=" * 80)
    print("Sentence", sentence_number)

    for word, gold, pred in zip(
        result["words"],
        result["gold_tags"],
        result["predicted_tags"]
    ):
        print(
            f"{word:20s} Gold={gold:6s} Pred={pred}"
        )

## 3. Evaluation using Seqeval

The gold tag sequence is compared with the Viterbi-predicted tag sequence.

We report precision, recall and F1 using `seqeval`.


In [ ]:
print("SEQEVAL CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        all_gold_tags,
        all_predicted_tags
    )
)

In [ ]:
precision = precision_score(
    all_gold_tags,
    all_predicted_tags
)

recall = recall_score(
    all_gold_tags,
    all_predicted_tags
)

f1 = f1_score(
    all_gold_tags,
    all_predicted_tags
)

seqeval_accuracy = accuracy_score(
    all_gold_tags,
    all_predicted_tags
)

print("Evaluation Results")
print("=" * 40)
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"Seqeval Accuracy : {seqeval_accuracy:.4f}")

In [ ]:
# Token-level accuracy for additional reference
gold_flat = [
    tag
    for sentence in all_gold_tags
    for tag in sentence
]

pred_flat = [
    tag
    for sentence in all_predicted_tags
    for tag in sentence
]

correct = sum(
    gold == pred
    for gold, pred in zip(gold_flat, pred_flat)
)

total = len(gold_flat)

token_accuracy = correct / total

print("Total tokens:", total)
print("Correct tokens:", correct)
print(f"Token Accuracy: {token_accuracy:.4f}")

In [ ]:
# Save Viterbi predictions
OUTPUT_FILE = "/content/HMM_NP_predictions.txt"

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for result in viterbi_results:
        for word, pred_tag in zip(
            result["words"],
            result["predicted_tags"]
        ):
            f.write(f"{word}\t{pred_tag}\n")

        f.write("\n")

print("Saved:", OUTPUT_FILE)

## 4. Confusion Matrix

This optional section shows token-level confusion between the three tags.


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    gold_flat,
    pred_flat,
    labels=tags
)

cm_df = pd.DataFrame(
    cm,
    index=tags,
    columns=tags
)

print(cm_df)

In [ ]:
plt.figure(figsize=(6, 5))

plt.imshow(cm)

plt.xticks(range(len(tags)), tags)
plt.yticks(range(len(tags)), tags)

plt.xlabel("Predicted Tag")
plt.ylabel("Actual Tag")
plt.title("HMM Noun Phrase Tagging - Confusion Matrix")

for i in range(len(tags)):
    for j in range(len(tags)):
        plt.text(
            j, i, cm[i, j],
            ha="center",
            va="center"
        )

plt.colorbar()
plt.show()

In [ ]:
# Final summary
print("=" * 70)
print("FINAL HMM NP TAGGING RESULTS")
print("=" * 70)

print("Training sentences :", len(train_data))
print("Test sentences     :", len(test_data))
print("Vocabulary size    :", vocab_size)
print("Tags               :", tags)
print("Add-K value        :", K)

print("\nEvaluation:")
print(f"Precision          : {precision:.4f}")
print(f"Recall             : {recall:.4f}")
print(f"F1 Score           : {f1:.4f}")
print(f"Token Accuracy     : {token_accuracy:.4f}")

print("\nGenerated files:")
print("1.", FORWARD_FILE)
print("2.", OUTPUT_FILE)

## Assignment Mapping

| Requirement | Implemented in |
|---|---|
| Read training/test data | Cells 4–6 |
| HMM training | Cells 8–10 |
| Initial probabilities | Cell 8 |
| Transition probabilities | Cell 9 |
| Emission probabilities | Cell 10 |
| Add-K smoothing | Cells 11–13 |
| Forward algorithm | Cells 14–18 |
| Probability of every test sentence | Cells 18–20 |
| Viterbi decoding | Cells 21–24 |
| Predict B-NP/I-NP/O | Cells 23–24 |
| Seqeval evaluation | Cells 25–26 |
| Precision/Recall/F1 | Cell 26 |
| Confusion matrix | Cells 29–30 |
| Save predictions | Cell 27 |

### Important interpretation

**Forward Algorithm:** computes the total probability of a sentence by considering all possible tag sequences.

**Viterbi:** finds the single most probable tag sequence for the sentence.

**Add-K smoothing:** prevents zero probabilities for unseen transitions and emissions.
